# Lab 1 — Del notebook caótico al proyecto reproducible

**Taller: MLOps en la práctica — del notebook a producción** · Universidad Nacional de Ingeniería
**Duración:** ~60 min · **Modalidad:** guiado + retos

## 🎯 Objetivos
Al terminar este lab podrás:
1. Identificar por qué un notebook "que funciona" **no** es un proyecto de ML confiable.
2. Hacer tu trabajo **reproducible**: seeds, versiones pinneadas, datos validados.
3. Encapsular todo el preprocesamiento + modelo en un **Pipeline de scikit-learn** (el corazón de un modelo desplegable).
4. Guardar el modelo como artefacto versionable.

## 📖 El caso: AndesTel
Trabajas en el equipo de datos de **AndesTel**, una telco peruana. El área comercial pierde ~14% de clientes por trimestre y quiere un modelo que prediga qué clientes están por irse (**churn**) para retenerlos con ofertas.

Un practicante dejó un "modelo que funciona" en un notebook... y hoy nadie puede reproducir sus resultados. Nos pasa a todos. Vamos a arreglarlo con método.

In [ ]:
# === Setup: instala dependencias (en Colab tarda ~1 min; en local usa tu venv) ===
%pip install -q scikit-learn==1.8.0 pandas pyarrow joblib pyyaml

In [ ]:
# === Carga de datos ===
# Opción A (recomendada): el instructor compartió una URL del repo del taller.
# Opción B: sube el archivo churn_telco_peru.csv manualmente (en Colab: icono de carpeta -> subir).
import os
import pandas as pd

URL_DATOS = ""  # <-- el instructor pega aquí la URL raw de GitHub, ej: https://raw.githubusercontent.com/<usuario>/taller-mlops-uni/main/data/churn_telco_peru.csv

if os.path.exists("churn_telco_peru.csv"):
    df = pd.read_csv("churn_telco_peru.csv")
elif URL_DATOS:
    df = pd.read_csv(URL_DATOS)
    df.to_csv("churn_telco_peru.csv", index=False)
else:
    raise FileNotFoundError("Sube churn_telco_peru.csv o define URL_DATOS")

print(df.shape)
df.head(3)

## 1. El anti-patrón: así se ve el "modelo del practicante"

Léelo con ojo crítico. **¿Cuántos problemas encuentras?** (hay al menos 6)

In [ ]:
# ⚠️ ANTI-EJEMPLO — NO imitar (pero sí ejecutar, para ver que "funciona")
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

df2 = df.dropna()                                  # ¿?
df2 = df2.drop(columns=["id_cliente"])
df2 = pd.get_dummies(df2)                          # ¿?
X = df2.drop(columns=["churn"])
y = df2["churn"]
X_train, X_test, y_train, y_test = train_test_split(X, y)   # ¿?
m = RandomForestClassifier()                        # ¿?
m.fit(X_train, y_train)
print("accuracy:", m.score(X_test, y_test))         # ¿?

### 🗣️ Discusión (5 min, en grupos de la sala Zoom)

Problemas del anti-ejemplo — compáralos con tu lista:

| # | Problema | Consecuencia en producción |
|---|---|---|
| 1 | Sin `random_state` en split ni modelo | Cada ejecución da un resultado distinto → irreproducible |
| 2 | `dropna()` bota filas enteras | En producción los clientes nuevos **sí** tendrán nulos, ¿y entonces? |
| 3 | `get_dummies` fuera de un pipeline | Si llega una categoría nueva (o falta una), las columnas no calzan y el modelo revienta |
| 4 | No se detectaron **duplicados** | Fuga de información: el mismo cliente puede caer en train y test |
| 5 | `accuracy` con clases desbalanceadas (14% churn) | Un modelo que dice "nadie se va" ya tiene 86% de accuracy |
| 6 | Nada queda guardado (modelo, versiones, parámetros) | "¿Con qué librería lo entrenaste?" → nadie sabe |

**💡 Para tu trabajo:** este anti-patrón es el estado real de muchos proyectos en empresas. El valor de MLOps empieza por eliminar estos 6 problemas de forma sistemática, no heroica.

## 2. Reproducibilidad: la regla de oro

> *Mismo código + mismos datos + misma configuración = mismo resultado. Siempre. En cualquier máquina.*

Tres pilares que aplicaremos ahora: **(a)** semillas fijas y configuración centralizada, **(b)** validación de datos a la entrada, **(c)** preprocesamiento dentro del pipeline.

In [ ]:
# === 2a. Configuración centralizada (en un proyecto real: params.yaml versionado en Git) ===
import yaml

CONFIG = """
data:
  path: churn_telco_peru.csv
  target: churn
  test_size: 0.2
model:
  tipo: random_forest
  n_estimators: 300
  max_depth: 8
  class_weight: balanced
random_state: 42
"""
with open("params.yaml", "w") as f:
    f.write(CONFIG)

params = yaml.safe_load(CONFIG)
params

**¿Por qué un archivo de configuración y no números sueltos en el código?**
Porque la configuración *es parte del experimento*. Si vive en un archivo versionado en Git, cada resultado es rastreable a la configuración exacta que lo produjo. Mañana lo conectaremos con MLflow para que esto sea automático.

In [ ]:
# === 2b. Validación de datos a la entrada (contrato de datos mínimo) ===
# Antes de entrenar, verificamos que los datos cumplen lo que esperamos.
# En producción esto se hace con Great Expectations o Pandera; aquí, la versión esencial:

def validar_datos(df: pd.DataFrame) -> pd.DataFrame:
    errores = []
    columnas_esperadas = {
        "id_cliente", "edad", "departamento", "plan", "tipo_contrato",
        "meses_antiguedad", "cargo_mensual_soles", "gb_datos_mes",
        "minutos_llamadas_mes", "lineas_adicionales", "tickets_soporte_6m",
        "caidas_servicio_mes", "dias_ultimo_pago_vencido", "factura_electronica", "churn",
    }
    if faltan := columnas_esperadas - set(df.columns):
        errores.append(f"Faltan columnas: {faltan}")
    if not df["churn"].isin([0, 1]).all():
        errores.append("churn debe ser binario")
    if (df["cargo_mensual_soles"] < 0).any():
        errores.append("cargos negativos")
    if (df["meses_antiguedad"] < 1).any():
        errores.append("antigüedad < 1 mes")
    if errores:
        raise ValueError("Datos inválidos: " + "; ".join(errores))

    n0 = len(df)
    df = df.drop_duplicates(subset="id_cliente", keep="first")
    print(f"✅ Validación OK | duplicados eliminados: {n0 - len(df)} | filas: {len(df)}")
    print(f"   Nulos restantes -> edad: {df.edad.isna().sum()}, gb_datos_mes: {df.gb_datos_mes.isna().sum()} (los imputará el pipeline)")
    return df

df = validar_datos(df)

**Nota clave:** eliminamos duplicados (eso sí es un error de datos), pero **NO** eliminamos los nulos. Los nulos existirán también en producción, así que la *imputación debe ser parte del modelo* — eso lo garantiza el pipeline.

In [ ]:
# === 2c. Split reproducible y estratificado ===
from sklearn.model_selection import train_test_split

FEATURES_NUM = ["edad", "meses_antiguedad", "cargo_mensual_soles", "gb_datos_mes",
                "minutos_llamadas_mes", "lineas_adicionales", "tickets_soporte_6m",
                "caidas_servicio_mes", "dias_ultimo_pago_vencido", "factura_electronica"]
FEATURES_CAT = ["departamento", "plan", "tipo_contrato"]
TARGET = params["data"]["target"]

X = df[FEATURES_NUM + FEATURES_CAT]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=params["data"]["test_size"],
    random_state=params["random_state"],
    stratify=y,                      # mantiene la proporción de churn en ambos sets
)
print(f"train: {X_train.shape} ({y_train.mean():.1%} churn) | test: {X_test.shape} ({y_test.mean():.1%} churn)")

## 3. El Pipeline: tu modelo ES el preprocesamiento + el estimador

Este es **el concepto más importante del día**. Un modelo desplegable no es solo el `RandomForest`: es *todo el camino desde el dato crudo hasta la predicción*. Si la imputación y el encoding viven dentro del pipeline:

- Se ajustan **solo con train** (adiós fuga de información).
- En producción, el servicio recibe datos crudos y el pipeline hace todo — imposible que el preprocesamiento de producción "se desvíe" del de entrenamiento (*training-serving skew*).
- Se guarda y versiona como **una sola pieza**.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

preprocesador = ColumnTransformer([
    ("num", Pipeline([
        ("imputar", SimpleImputer(strategy="median")),
        ("escalar", StandardScaler()),
    ]), FEATURES_NUM),
    ("cat", Pipeline([
        ("imputar", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),  # categoría nueva en prod -> no revienta
    ]), FEATURES_CAT),
])

modelo = Pipeline([
    ("preproc", preprocesador),
    ("clf", RandomForestClassifier(
        n_estimators=params["model"]["n_estimators"],
        max_depth=params["model"]["max_depth"],
        class_weight=params["model"]["class_weight"],
        random_state=params["random_state"],
        n_jobs=-1,
    )),
])

modelo.fit(X_train, y_train)
modelo

In [ ]:
# === 4. Evaluación honesta (no accuracy a secas) ===
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, classification_report

proba = modelo.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

metricas = {
    "roc_auc": round(roc_auc_score(y_test, proba), 4),
    "f1": round(f1_score(y_test, pred), 4),
    "precision": round(precision_score(y_test, pred), 4),
    "recall": round(recall_score(y_test, pred), 4),
}
print(metricas)
print()
print(classification_report(y_test, pred, target_names=["se queda", "churn"]))

### 🗣️ Pregunta de negocio (2 min)

Retener a un cliente cuesta S/ 15 (una llamada + oferta). Perderlo cuesta en promedio S/ 540 al año. **¿Te conviene un modelo con más precision o más recall? ¿Moverías el umbral de 0.5?**

*(Pista: el costo de un falso negativo ≈ 36× el de un falso positivo → conviene bajar el umbral y priorizar recall. Esta conversación con negocio es parte del trabajo de MLOps: el umbral es un parámetro de negocio, no un default.)*

In [ ]:
# === 5. Persistencia del modelo + metadatos (versión artesanal de lo que MLflow hará mañana) ===
import joblib, json, platform, sklearn
from datetime import datetime, timezone

joblib.dump(modelo, "modelo_churn_v1.joblib")

metadata = {
    "fecha_entrenamiento": datetime.now(timezone.utc).isoformat(),
    "metricas_test": metricas,
    "parametros": params,
    "features_num": FEATURES_NUM,
    "features_cat": FEATURES_CAT,
    "versiones": {
        "python": platform.python_version(),
        "scikit-learn": sklearn.__version__,
        "pandas": pd.__version__,
    },
    "filas_entrenamiento": len(X_train),
}
with open("modelo_churn_v1.metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Guardado: modelo_churn_v1.joblib + metadata")
print(json.dumps(metadata["versiones"], indent=2))

In [ ]:
# Prueba de fuego: ¿el modelo cargado predice sobre datos CRUDOS (con nulos y categorías)?
modelo_cargado = joblib.load("modelo_churn_v1.joblib")

cliente_nuevo = pd.DataFrame([{
    "edad": None,                      # ¡nulo! el pipeline lo imputa
    "departamento": "Cusco",
    "plan": "Prepago",
    "tipo_contrato": "Mensual",
    "meses_antiguedad": 3,
    "cargo_mensual_soles": 29.9,
    "gb_datos_mes": 12.5,
    "minutos_llamadas_mes": 180,
    "lineas_adicionales": 0,
    "tickets_soporte_6m": 4,
    "caidas_servicio_mes": 3,
    "dias_ultimo_pago_vencido": 12,
    "factura_electronica": 0,
}])
p = modelo_cargado.predict_proba(cliente_nuevo)[0, 1]
print(f"Probabilidad de churn del cliente nuevo: {p:.1%}  {'🚨 llamar a retención' if p > 0.35 else '✅ bajo riesgo'}")

## 🎯 Retos (15 min)

**Reto 1 (todos):** cambia `random_state` a otro valor en `params.yaml`, re-ejecuta desde el split y compara métricas. Luego vuelve a 42 y verifica que recuperas **exactamente** los mismos números. Eso es reproducibilidad.

**Reto 2 (todos):** el negocio pide priorizar recall. Encuentra el umbral que logre **recall ≥ 0.60** con la mejor precision posible. Pista: `from sklearn.metrics import precision_recall_curve`.

**Reto 3 (avanzado):** agrega al `validar_datos` una verificación de rangos para `edad` (18–100) y una alerta (no error) si la tasa de churn del dataset se aleja más de 5 puntos de la histórica (13.9%).

**Reto 4 (avanzado):** crea una feature nueva `cargo_por_gb = cargo_mensual_soles / (gb_datos_mes + 1)` **dentro del pipeline** usando `FunctionTransformer`, para que también se aplique en producción.

## 📌 Lo que te llevas de este lab

- ✅ La regla de oro: mismo código + datos + config = mismo resultado.
- ✅ Validar datos a la entrada; los nulos se imputan **dentro** del pipeline, no se botan.
- ✅ El artefacto desplegable es el **Pipeline completo**, no el estimador.
- ✅ Métricas alineadas al problema (desbalance → AUC/recall, no accuracy) y umbral como decisión de negocio.
- ✅ Todo modelo guardado lleva metadatos: fecha, métricas, config, versiones.

**Problema pendiente:** entrenamos 1 modelo. Mañana entrenaremos 20 variantes... ¿cómo no perdernos? → **Lab 2: MLflow**.